# rlmflow visualization walkthrough

A guided tour of every visualization that ships with rflow. Everything here renders **inline in Jupyter** and runs **offline** against the saved word-search run under `examples/_runs/word-search/baseline/` — no API keys, no LLM calls.

Every viewer / viz function takes a unified **`source`**: a single `Graph`, an iterable of `Graph`s, a `Trace`, or a path to a run directory / `graph.json` / `trace.json`. So you can hand any of these the same run-directory path string and skip the loading boilerplate. For the underlying `Graph` query API (walk, find, where, session), see [`node_basics.ipynb`](./node_basics.ipynb).

**What we cover**

1. Loading a run
2. Inline tree rendering — `graph.tree()`, `graph_tree`
3. **Interactive viewer** — `open_viewer` (Gradio: clickable graph, slider, per-agent chat)
4. Plotly graph + HTML stepper — `graph_plot`, `save_html`, `save_steps`
5. Topology renders — Mermaid (state / flowchart / sequence), DOT, D2
6. Step-indexed timeline — `gantt`, `gantt_html`
7. Per-state detail — `code_log`, `error_summary`, `agent_transcript`
8. Reports and token usage — `report_md`, `token_sparkline`
9. Any snapshot list is a source — slicing the trace
10. CLI equivalents

## 1. Load a run

`rflow.Graph.load(path)` rebuilds the final `Graph`; `graph.trace()` retraces the run as one `Graph` per tick. Most viz functions also take the run-directory path directly — we'll mix both styles below to show the unified `source` API.

In [ ]:
import rflow

RUN = "../_runs/word-search/baseline"
final = rflow.Graph.load(RUN)
graphs = final.trace().graphs
print(f"loaded {len(graphs)} snapshots  \u00b7  final has {len(final.agents)} agents")

## 2. Inline tree rendering

`graph.tree()` returns a per-agent timeline as plain text. `graph_tree(source)` is the same render reachable from the unified `source` API, so you can point it straight at a run-directory path.

In [ ]:
print(final.tree())

In [ ]:
from rflow.utils.viewer import graph_tree

# `source` accepts a Graph, path, Trace, or list[Graph] — try the path:
print(graph_tree(RUN))

## 3. Interactive viewer

The full interactive graph + per-agent chat lives in `open_viewer`. It embeds inline in Jupyter via Gradio, with a step slider, clickable nodes, and a color-coded conversation panel for any agent you click.

For static export (PR comments, email), use the Plotly stepper in section 4 or the topology renders in section 5.

In [ ]:
from rflow.utils.viewer import open_viewer

# `open_viewer` takes the same `source` — a run path, graph, or list of graphs.
open_viewer(
    RUN,
    inline=True,
    height=720,
    quiet=True,
)

## 4. Plotly graph + HTML stepper

`graph_plot(source)` builds a Plotly figure with every node laid out as a tidy tree (one column per branching child, edges from `Graph.edges`). `save_html(source, out)` stitches every snapshot into a self-contained HTML page with arrow-key navigation; `save_steps(source, out_dir)` writes one PNG per step (handy for slide decks). All take the unified `source`.

In [ ]:
from rflow.utils.viewer import graph_plot

graph_plot(final, height=500)

In [ ]:
import tempfile
from pathlib import Path
from IPython.display import IFrame
from rflow.utils.viewer import save_html

out = Path(tempfile.gettempdir()) / "rlmflow-stepper.html"
save_html(RUN, out)
print(f"wrote {out}")
IFrame(str(out), width="100%", height=720)

## 5. Topology renders

Static topology renders for embedding in docs, PRs, post-mortems. Mermaid blocks render inline via `IPython.display.Markdown`. The same renders are reachable from the CLI as `rlmflow render <path> -f F`.

In [ ]:
from IPython.display import Markdown
from rflow.utils.export import to_mermaid

Markdown(f"```mermaid\n{to_mermaid(final)}\n```")

In [ ]:
from rflow.utils.export import to_mermaid_flowchart

Markdown(f"```mermaid\n{to_mermaid_flowchart(final)}\n```")

In [ ]:
from rflow.utils.export import to_mermaid_sequence

Markdown(f"```mermaid\n{to_mermaid_sequence(final)}\n```")

In [ ]:
from rflow.utils.export import to_dot, to_d2

print("--- DOT (paste into a Graphviz renderer) ---")
print(to_dot(final))
print()
print("--- D2 (https://d2lang.com) ---")
print(to_d2(final))

## 6. Step-indexed timeline

`gantt(graphs)` prints a Rich table — one row per agent, one column per step, single-letter glyphs (`Q`/`A`/`O`/`S`/`R`/`F`/`E`). `gantt_html(graphs)` writes a colorful self-contained HTML page; we render it inline below.

In [ ]:
from rflow.utils.viz import gantt

gantt(RUN)

In [ ]:
from IPython.display import HTML
from rflow.utils.viz import gantt_html

HTML(gantt_html(RUN, title="word-search run"))

## 7. Per-state detail

Drill into the actual code that ran, errors that happened, and the chat-log view for any agent. (This run has a couple of real `error_output` nodes, so `error_summary` shows grouped output.)

In [ ]:
from rflow.utils.viz import code_log

print(code_log(final))

In [ ]:
# `error_summary` groups errors by kind across the whole subtree.
from rflow.utils.viz import error_summary

print(error_summary(RUN))

In [ ]:
# `agent_transcript` renders one agent's sub-Graph as a chat log
# (also available as `final["root.cols"].transcript()`).
from rflow.utils.viewer import agent_transcript

stream = agent_transcript(final["root.cols"])
print(stream[:1800] + ("\n..." if len(stream) > 1800 else ""))

In [ ]:
# `graph_session` is the flat, every-agent chat log in graph order
# (also available as `final.session()`).
from rflow.utils.viewer import graph_session

flat = graph_session(RUN)
print(f"flat session log: {len(flat):,} chars\n")
print(flat[:1500] + "\n...")

## 8. Token Usage & Reports

`report_md(source)` bundles tree + result + errors into one Markdown report. `token_sparkline(source)` shows cumulative token usage across snapshots, and raw token totals are a query-API call away via `graph.tokens()`.

In [ ]:
from rflow.utils.viz import token_sparkline

print(token_sparkline(graphs))

inp, out = final.tokens()
print(f"tokens: {inp + out:,} total  ({inp:,} in / {out:,} out)")

In [ ]:
from rflow.utils.viz import report_md

Markdown(report_md(RUN, title="word-search run"))

## 9. Any snapshot list is a source

Because the unified `source` accepts a `list[Graph]`, you can render any slice of a run — e.g. just the first half of the timeline — without re-loading or re-running anything.

In [ ]:
# A list[Graph] is a valid `source` — render a gantt for just the first
# half of the run by slicing the retraced snapshots.
half = graphs[: len(graphs) // 2 + 1]
print(f"full run: {len(graphs)} steps  \u00b7  slice: {len(half)} steps\n")
gantt(half)

## 10. CLI equivalents

Every render here is also available without writing Python. From a shell:

```bash
rlmflow view   examples/_runs/word-search/baseline
rlmflow render examples/_runs/word-search/baseline -f mermaid-flowchart
rlmflow render examples/_runs/word-search/baseline -f gantt-html  -o gantt.html
rlmflow render examples/_runs/word-search/baseline -f report-md   -o report.md
rlmflow render examples/_runs/word-search/baseline -f code-log
rlmflow render examples/_runs/word-search/baseline -f tree
rlmflow render examples/_runs/word-search/baseline -f ascii-boxes
rlmflow render examples/_runs/word-search/baseline -f tokens
rlmflow render examples/_runs/word-search/baseline -f steps -o frames/ --marker-mult 3.5 --text-mult 2.2
```

All formats: `mermaid` · `mermaid-flowchart` · `mermaid-sequence` · `dot` · `d2` · `tree` · `ascii-boxes` · `gantt-html` · `report-md` · `code-log` · `error-summary` · `tokens` · `html` · `image` · `steps`.

Figure formats (`html`, `image`, `steps`) accept `--element-mult`, `--marker-mult`, `--text-mult`, `--normalize-labels`, and `--no-normalize-labels`.